In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv(".env.local", override=False)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = os.getenv("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = os.getenv("TF_FORCE_GPU_ALLOW_GROWTH", "true")
os.environ["WANDB_SILENT"] = os.getenv("WANDB_SILENT", "true")

In [7]:
import wandb
import numpy as np
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow import keras as k

In [8]:
wandb_key = os.getenv("WANDB_API_KEY")
if wandb_key and wandb_key != "your_wandb_api_key_here":
    wandb.login(key=wandb_key)
else:
    wandb.login()

In [9]:
import wandb
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint
import numpy as np


def env_int(name, default):
    return int(os.getenv(name, default))


def env_float(name, default):
    return float(os.getenv(name, default))


class LogLRCallback(k.callbacks.Callback):
    """Log optimizer learning rate each epoch."""

    def on_epoch_end(self, epoch, logs=None):
        lr = self.model.optimizer.learning_rate
        lr_val = float(lr.numpy() if hasattr(lr, "numpy") else lr)
        wandb.log({"lr": lr_val}, step=self.model.optimizer.iterations.numpy())


class LogSamplesCallback(k.callbacks.Callback):
    """Log a small table of predictions and images every epoch."""

    def __init__(self, x, y, labels, max_rows=32):
        super().__init__()
        self.x = x[:max_rows]
        self.y = y[:max_rows]
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x, verbose=0)
        y_true = np.argmax(self.y, axis=1)
        y_pred = np.argmax(preds, axis=1)

        table = wandb.Table(columns=["image", "y_true", "y_pred", "correct", "p(y_pred)"])
        for i in range(len(self.x)):
            table.add_data(
                wandb.Image(self.x[i]),
                self.labels[y_true[i]],
                self.labels[y_pred[i]],
                bool(y_true[i] == y_pred[i]),
                float(np.max(preds[i])),
            )
        wandb.log({f"samples/epoch_{epoch + 1}": table})


class ConfusionMatrixCallback(k.callbacks.Callback):
    """Log a confusion matrix from the full validation set each epoch."""

    def __init__(self, x_val, y_val, labels):
        super().__init__()
        self.x_val = x_val
        self.y_val = y_val
        self.labels = labels

    def on_epoch_end(self, epoch, logs=None):
        preds = self.model.predict(self.x_val, verbose=0)
        y_true = np.argmax(self.y_val, axis=1)
        y_pred = np.argmax(preds, axis=1)
        cm_plot = wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.labels,
        )
        wandb.log({"confusion_matrix": cm_plot})


class CIFAR10Trainer:
    def __init__(self, project_name=None, run_name=None):
        self.cfg = dict(
            dropout=env_float("MODEL_DROPOUT", 0.3),
            conv_1_filters=env_int("MODEL_CONV1_FILTERS", 32),
            conv_2_filters=env_int("MODEL_CONV2_FILTERS", 64),
            dense_size=env_int("MODEL_DENSE_SIZE", 128),
            learn_rate=env_float("OPT_LR", 1e-3),
            epochs=env_int("TRAIN_EPOCHS", 6),
            batch_size=env_int("TRAIN_BATCH_SIZE", 64),
            sample=env_int("DATASET_SAMPLE_SIZE", 20000),
        )

        self.run = wandb.init(
            project=project_name or os.getenv("WANDB_PROJECT", "Lab2-cifar10-tracking"),
            name=run_name or os.getenv("WANDB_RUN_NAME", "cifar10_cnn_env"),
            config=self.cfg,
            mode=os.getenv("WANDB_MODE", "online"),
            settings=wandb.Settings(start_method="thread"),
        )
        self.config = wandb.config
        self.labels = [
            "airplane",
            "automobile",
            "bird",
            "cat",
            "deer",
            "dog",
            "frog",
            "horse",
            "ship",
            "truck",
        ]
        self._prepare_data()

    def _prepare_data(self):
        (xtr, ytr), (xte, yte) = cifar10.load_data()
        ytr = ytr.squeeze()
        yte = yte.squeeze()

        n = min(int(self.config.sample), len(xtr), len(xte))
        xtr = xtr[:n].astype("float32") / 255.0
        ytr = ytr[:n]
        xte = xte[:n].astype("float32") / 255.0
        yte = yte[:n]

        self.X_train = xtr
        self.X_test = xte
        self.y_train = to_categorical(ytr, num_classes=10)
        self.y_test = to_categorical(yte, num_classes=10)
        self.num_classes = self.y_test.shape[1]

    def _build_model(self):
        inputs = k.Input(shape=(32, 32, 3))
        x = k.layers.RandomFlip("horizontal")(inputs)
        x = k.layers.RandomRotation(0.05)(x)

        x = k.layers.Conv2D(self.config.conv_1_filters, (3, 3), padding="same", activation="relu")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Conv2D(self.config.conv_1_filters, (3, 3), padding="same", activation="relu")(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        x = k.layers.Conv2D(self.config.conv_2_filters, (3, 3), padding="same", activation="relu")(x)
        x = k.layers.BatchNormalization()(x)
        x = k.layers.Conv2D(self.config.conv_2_filters, (3, 3), padding="same", activation="relu")(x)
        x = k.layers.MaxPooling2D((2, 2))(x)
        x = k.layers.Dropout(self.config.dropout)(x)

        x = k.layers.GlobalAveragePooling2D()(x)
        x = k.layers.Dense(self.config.dense_size, activation="relu")(x)
        x = k.layers.Dropout(self.config.dropout)(x)
        outputs = k.layers.Dense(self.num_classes, activation="softmax")(x)

        model = k.Model(inputs, outputs)
        opt = k.optimizers.Adam(learning_rate=self.config.learn_rate)
        model.compile(loss="categorical_crossentropy", optimizer=opt, metrics=["accuracy"])
        return model

    def _log_model_artifact(self, model):
        summary_lines = []
        model.summary(print_fn=summary_lines.append)
        summary_txt = "\n".join(summary_lines)

        os.makedirs("artifacts", exist_ok=True)
        with open("artifacts/model_summary.txt", "w") as f:
            f.write(summary_txt)

        model_path = "artifacts/model.keras"
        model.save(model_path)

        art = wandb.Artifact("cifar10_model", type="model")
        art.add_file("artifacts/model_summary.txt")
        art.add_file(model_path)
        self.run.log_artifact(art)

    def train(self):
        model = self._build_model()
        os.makedirs("checkpoints", exist_ok=True)

        callbacks = [
            WandbMetricsLogger(log_freq=10),
            WandbModelCheckpoint("checkpoints/cifar10-model-{epoch:02d}.keras", save_weights_only=False),
            LogLRCallback(),
            LogSamplesCallback(self.X_test, self.y_test, self.labels, max_rows=32),
            ConfusionMatrixCallback(self.X_test, self.y_test, self.labels),
        ]

        model.fit(
            self.X_train,
            self.y_train,
            validation_data=(self.X_test, self.y_test),
            epochs=self.config.epochs,
            batch_size=self.config.batch_size,
            callbacks=callbacks,
            verbose=1,
        )

        loss, acc = model.evaluate(self.X_test, self.y_test, verbose=0)
        wandb.log({"final/loss": loss, "final/accuracy": acc})

        self._log_model_artifact(model)
        self.run.finish()


CIFAR10Trainer().train()

Epoch 1/6
157/157 ━━━━━━━━━━━━━━━━━━━━ 53s 329ms/step - accuracy: 0.2875 - loss: 1.9016 - val_accuracy: 0.1471 - val_loss: 2.7663
Epoch 2/6
157/157 ━━━━━━━━━━━━━━━━━━━━ 64s 411ms/step - accuracy: 0.3847 - loss: 1.6449 - val_accuracy: 0.1540 - val_loss: 3.2933
Epoch 3/6
157/157 ━━━━━━━━━━━━━━━━━━━━ 59s 373ms/step - accuracy: 0.4440 - loss: 1.4975 - val_accuracy: 0.2049 - val_loss: 2.3704
Epoch 4/6
157/157 ━━━━━━━━━━━━━━━━━━━━ 65s 413ms/step - accuracy: 0.4786 - loss: 1.4083 - val_accuracy: 0.3697 - val_loss: 1.9187
Epoch 5/6
157/157 ━━━━━━━━━━━━━━━━━━━━ 57s 364ms/step - accuracy: 0.5089 - loss: 1.3507 - val_accuracy: 0.4672 - val_loss: 1.4239
Epoch 6/6
157/157 ━━━━━━━━━━━━━━━━━━━━ 62s 393ms/step - accuracy: 0.5268 - loss: 1.2899 - val_accuracy: 0.4720 - val_loss: 1.4934
